# WeatherGPT — Image-Based Weather Classifier (Training Notebook)
**SIH26068 · Ministry of Earth Sciences · ML/NLP layer extension**

This notebook trains a CNN (transfer learning) that classifies the **current weather condition
from a photo** — e.g. a citizen points their camera at the sky and gets an instant weather
label (`rain`, `lightning`, `hail`, `sandstorm`, `snow`, `fog/smog`, `sunrise`, `shine`, `frost`,
`glaze`, `dew`, `rainbow`). This is a genuinely useful addition to WeatherGPT:

- **Offline-capable**: exported to TFLite, it runs fully on-device — works even with zero
  connectivity, which matters a lot for a disaster-alert tool.
- **Ground-truth signal**: citizen photo reports can cross-check/supplement official alert
  feeds (e.g. corroborating a hailstorm report in a region before it's officially confirmed).
- Fits cleanly into the existing architecture as an addition to the **ML/NLP layer**
  (Members 3 & 4) — either served on-device via `tflite_flutter` in the mobile app (Member 1),
  or behind a new backend endpoint (Member 5/6) using the FastAPI stub included alongside
  this notebook (`serve_model.py`).

## Before you start
1. In Colab: **Runtime → Change runtime type → T4 GPU** (training on CPU will be painfully slow).
2. You'll need a free Kaggle account + API token (`kaggle.json`) to download the dataset —
   instructions are in Section 2.
3. Expect ~25–40 minutes total training time on a T4 for the full two-phase schedule below.

## Dataset
**Weather Dataset** (Kaggle: `jehanbhathena/weather-dataset`) — 6,862 real-world images across
11 classes: `dew, fogsmog, frost, glaze, hail, lightning, rain, rainbow, rime, sandstorm, shine, snow, sunrise`.
(That's 13 folders in the raw download — a couple of near-duplicate categories like `shine`/`sunrise`
are kept separate on purpose; merge them in Section 3 if your use case doesn't need the distinction.)

*Faster baseline alternative*: if you're short on time before demo day, the **Multi-class Weather
Dataset** (`pratik2901/multiclass-weather-dataset`, ~1,125 images, 4 classes: Cloudy/Rain/Shine/Sunrise)
trains in a fraction of the time with fewer, cleaner classes. Swap the `KAGGLE_DATASET` variable
in Section 2 and everything downstream still works — the pipeline is class-count-agnostic.


## 1 — Environment setup

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

# NOTE: scikit-learn, seaborn, and matplotlib are NOT upgraded here on purpose —
# Colab already ships working versions of all three, and force-upgrading a package
# that's already loaded in the running kernel is exactly what causes the
# "cannot import name '_align_api_if_sparse'" ImportError some Colab environments hit.
# Only kagglehub/imagehash genuinely need installing (they're not preinstalled).
!pip -q install kagglehub

import os, shutil, pathlib, json, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
BACKBONE = "efficientnet"   # "efficientnet" (best accuracy) or "mobilenet" (fastest on-device)


## 2 — Download and combine multiple datasets

One dataset alone may not give enough examples per class, or may be missing classes you
care about — so this section is written to combine **any number** of Kaggle datasets into
one unified, deduplicated training set. Add a new entry to `DATASETS_TO_COMBINE` and a
mapping in `CLASS_MAPPINGS` to bring in a third, fourth, etc. dataset later without
rewriting anything else in this notebook.

Get a Kaggle API token first: Kaggle profile → **Settings → API → Create New Token** —
downloads `kaggle.json`. Upload it below when prompted.

In [ ]:
from google.colab import files
print("Upload your kaggle.json now:")
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip -q install imagehash
import kagglehub
import imagehash
from PIL import Image


In [ ]:
# --- Datasets to combine ---
# Add/remove entries here to change what goes into the combined training set.
DATASETS_TO_COMBINE = [
    "jehanbhathena/weather-dataset",              # 11 classes, ~6.9k images
    "pratik2901/multiclass-weather-dataset",       # 4 classes, ~1.1k images (adds more shine/rain/sunrise examples + a distinct 'cloudy' class)
]

# --- Class name harmonization ---
# Each source dataset uses its own folder-name conventions. Map every source class
# name to ONE unified target class name. This is the step that actually lets you
# combine datasets correctly instead of ending up with duplicate near-identical
# classes like "Shine" and "shine" as two separate labels.
CLASS_MAPPINGS = {
    "jehanbhathena/weather-dataset": {
        "dew": "dew", "fogsmog": "fogsmog", "frost": "frost", "glaze": "glaze",
        "hail": "hail", "lightning": "lightning", "rain": "rain", "rainbow": "rainbow",
        "rime": "rime", "sandstorm": "sandstorm", "shine": "shine", "snow": "snow",
        "sunrise": "sunrise",
    },
    "pratik2901/multiclass-weather-dataset": {
        "Cloudy": "cloudy",   # new class — not present in the first dataset
        "Rain": "rain", "Shine": "shine", "Sunrise": "sunrise",
    },
}

# Set MERGE_ICY_CLASSES = True to collapse frost/glaze/rime into one 'icy_conditions'
# class — these three are visually near-identical (see Section 11's confusion matrix
# in earlier runs) and merging them often gives a cleaner demo with fewer confusing errors.
MERGE_ICY_CLASSES = False
if MERGE_ICY_CLASSES:
    for mapping in CLASS_MAPPINGS.values():
        for k, v in list(mapping.items()):
            if v in ("frost", "glaze", "rime"):
                mapping[k] = "icy_conditions"

COMBINED_ROOT = "/content/combined_dataset"


In [ ]:
def find_data_root(path):
    """Locates the folder that directly contains class subfolders — Kaggle downloads
    sometimes nest an extra directory level."""
    for root, dirs, _ in os.walk(path):
        subdirs = [d for d in dirs if not d.startswith('.')]
        if len(subdirs) >= 2:
            sample_dir = os.path.join(root, subdirs[0])
            if os.path.isdir(sample_dir) and any(
                f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in os.listdir(sample_dir)
            ):
                return root
    return path


os.makedirs(COMBINED_ROOT, exist_ok=True)
dataset_stats = {}

for dataset_id in DATASETS_TO_COMBINE:
    print(f"Downloading {dataset_id} ...")
    path = kagglehub.dataset_download(dataset_id)
    data_root = find_data_root(path)
    mapping = CLASS_MAPPINGS[dataset_id]
    source_tag = dataset_id.split('/')[-1][:12]  # short tag to avoid filename collisions

    copied = 0
    for source_class, target_class in mapping.items():
        source_dir = os.path.join(data_root, source_class)
        if not os.path.isdir(source_dir):
            print(f"  ! '{source_class}' not found in {dataset_id}, skipping")
            continue
        target_dir = os.path.join(COMBINED_ROOT, target_class)
        os.makedirs(target_dir, exist_ok=True)
        for fname in os.listdir(source_dir):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            # Prefix with the source dataset tag so identical filenames from
            # different datasets never collide/overwrite each other.
            new_name = f"{source_tag}_{fname}"
            shutil.copy2(os.path.join(source_dir, fname), os.path.join(target_dir, new_name))
            copied += 1
    dataset_stats[dataset_id] = copied
    print(f"  copied {copied} images into the combined set")

print("\nPer-dataset contribution:", dataset_stats)
print("Combined classes:", sorted(os.listdir(COMBINED_ROOT)))


### Deduplicate across datasets

Combining datasets risks two problems if you skip this step: (1) the exact same stock
photo appearing in two datasets under the same class inflates that class's count without
adding real information, and (2) a near-duplicate photo ending up in both your training
and test splits leaks information and makes your test accuracy look better than it really
is. Perceptual hashing (`imagehash`) catches both exact and near-duplicate images.

In [ ]:
HASH_SIZE = 8            # larger = more sensitive to small differences
DEDUP_THRESHOLD = 4      # max Hamming distance to treat two images as duplicates

removed_total = 0
for class_name in sorted(os.listdir(COMBINED_ROOT)):
    class_dir = os.path.join(COMBINED_ROOT, class_name)
    seen_hashes = []
    files_in_class = os.listdir(class_dir)
    removed_in_class = 0

    for fname in files_in_class:
        fpath = os.path.join(class_dir, fname)
        try:
            h = imagehash.phash(Image.open(fpath), hash_size=HASH_SIZE)
        except Exception:
            os.remove(fpath)  # corrupt/unreadable image — drop it rather than let it crash training later
            continue

        is_duplicate = any((h - seen) <= DEDUP_THRESHOLD for seen in seen_hashes)
        if is_duplicate:
            os.remove(fpath)
            removed_in_class += 1
        else:
            seen_hashes.append(h)

    if removed_in_class:
        print(f"  {class_name}: removed {removed_in_class} duplicate/near-duplicate images")
    removed_total += removed_in_class

print(f"\nTotal duplicates removed: {removed_total}")

DATA_ROOT = COMBINED_ROOT
class_names = sorted(os.listdir(DATA_ROOT))
NUM_CLASSES = len(class_names)
print(f"\nFinal combined dataset: {NUM_CLASSES} classes at {DATA_ROOT}")
for c in class_names:
    print(f"  {c:16s}: {len(os.listdir(os.path.join(DATA_ROOT, c)))} images")


## 3 — Explore the combined dataset (always look at your data before training on it)

In [ ]:
counts = {c: len(os.listdir(os.path.join(DATA_ROOT, c))) for c in class_names}

plt.figure(figsize=(10, 4))
plt.bar(counts.keys(), counts.values(), color='#3E7CB1')
plt.xticks(rotation=45, ha='right')
plt.ylabel("Image count")
plt.title("Combined class distribution — check for imbalance before training")
plt.tight_layout()
plt.show()

for c, n in sorted(counts.items(), key=lambda x: x[1]):
    print(f"  {c:16s}: {n}")


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for ax, cname in zip(axes.flat, class_names):
    folder = os.path.join(DATA_ROOT, cname)
    fname = os.listdir(folder)[0]
    img = plt.imread(os.path.join(folder, fname))
    ax.imshow(img)
    ax.set_title(cname, fontsize=11)
    ax.axis('off')
for ax in axes.flat[len(class_names):]:
    ax.axis('off')
plt.tight_layout()
plt.show()


## 4 — Build the data pipeline

Split 70% train / 15% validation / 15% test. Using `image_dataset_from_directory` twice
(train vs. a combined val+test split), then splitting that second set in half by batch —
simple and avoids pulling in extra dependencies for a 2-week hackathon timeline.

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT, validation_split=0.3, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int',
)
val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT, validation_split=0.3, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int',
)

assert train_ds.class_names == val_test_ds.class_names
class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

val_batches = tf.data.experimental.cardinality(val_test_ds)
test_ds = val_test_ds.take(val_batches // 2)
val_ds = val_test_ds.skip(val_batches // 2)

print(f"Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Val batches:   {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"Test batches:  {tf.data.experimental.cardinality(test_ds).numpy()}")


In [ ]:
# Data augmentation — applied ONLY to the training set. Keeping this modest
# (not extreme rotations/crops) because weather cues like sky color and cloud
# texture are the actual signal; over-aggressive augmentation can wash that out.
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.12),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomBrightness(0.15),
], name="augmentation")

if BACKBONE == "efficientnet":
    preprocess_input = tf.keras.applications.efficientnet.preprocess_input
else:
    preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

AUTOTUNE = tf.data.AUTOTUNE

def prep_train(x, y):
    x = data_augmentation(x)
    x = preprocess_input(x)
    return x, y

def prep_eval(x, y):
    x = preprocess_input(x)
    return x, y

train_ds_p = train_ds.map(prep_train, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
val_ds_p = val_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
test_ds_p = test_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)


## 5 — Class weights (handles the imbalance visible in Section 3)

In [ ]:
all_train_labels = np.concatenate([y.numpy() for _, y in train_ds], axis=0)

class_weight_values = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(all_train_labels),
    y=all_train_labels,
)
class_weights = {i: w for i, w in enumerate(class_weight_values)}
print("Class weights:", {class_names[i]: round(w, 2) for i, w in class_weights.items()})


## 6 — Model: transfer learning

Training a CNN from scratch on ~6-7k images would badly overfit. Instead we start from an
ImageNet-pretrained backbone (already knows general visual features — edges, textures,
color gradients) and fine-tune it for weather classification.

- **EfficientNetB0** — best accuracy per parameter, used here as the primary model.
- **MobileNetV2** — smaller and faster; swap `BACKBONE = "mobilenet"` in Section 1 if the
  priority is on-device latency over the last percent of accuracy (still very strong).

In [ ]:
def build_model(num_classes, backbone="efficientnet", input_shape=(224, 224, 3)):
    if backbone == "efficientnet":
        base = tf.keras.applications.EfficientNetB0(
            include_top=False, weights='imagenet', input_shape=input_shape
        )
    elif backbone == "mobilenet":
        base = tf.keras.applications.MobileNetV2(
            include_top=False, weights='imagenet', input_shape=input_shape
        )
    else:
        raise ValueError(f"Unknown backbone: {backbone}")

    base.trainable = False  # Phase 1: freeze — train only the new head first

    inputs = tf.keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    return model, base

model, base_model = build_model(NUM_CLASSES, backbone=BACKBONE)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()


## 7 — Callbacks (stop overfitting, save the best checkpoint, adapt LR)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=6, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras', monitor='val_accuracy', save_best_only=True
    ),
    tf.keras.callbacks.CSVLogger('training_log.csv'),
]


## 8 — Phase 1: train the new head (backbone frozen)

In [ ]:
EPOCHS_PHASE1 = 15

history1 = model.fit(
    train_ds_p,
    validation_data=val_ds_p,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weights,
    callbacks=callbacks,
)


## 9 — Phase 2: fine-tune the top of the backbone

Unfreeze the last ~30% of the backbone's layers and continue training at a much lower
learning rate. This lets the model adapt higher-level ImageNet features (originally tuned
for objects like "dog", "car") toward weather-specific texture/color cues, without wrecking
the low-level features (edges, gradients) that are still useful.

In [ ]:
base_model.trainable = True

fine_tune_at = int(len(base_model.layers) * 0.7)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

EPOCHS_PHASE2 = 12
total_epochs = EPOCHS_PHASE1 + EPOCHS_PHASE2

history2 = model.fit(
    train_ds_p,
    validation_data=val_ds_p,
    epochs=total_epochs,
    initial_epoch=history1.epoch[-1] + 1,
    class_weight=class_weights,
    callbacks=callbacks,
)


## 10 — Training curves

In [ ]:
def merge_history(h1, h2, key):
    return h1.history[key] + h2.history[key]

acc = merge_history(history1, history2, 'accuracy')
val_acc = merge_history(history1, history2, 'val_accuracy')
loss = merge_history(history1, history2, 'loss')
val_loss = merge_history(history1, history2, 'val_loss')

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(acc, label='train'); ax[0].plot(val_acc, label='val')
ax[0].axvline(EPOCHS_PHASE1 - 1, color='gray', linestyle='--', label='fine-tuning starts')
ax[0].set_title('Accuracy'); ax[0].legend()
ax[1].plot(loss, label='train'); ax[1].plot(val_loss, label='val')
ax[1].axvline(EPOCHS_PHASE1 - 1, color='gray', linestyle='--')
ax[1].set_title('Loss'); ax[1].legend()
plt.tight_layout(); plt.show()


## 11 — Evaluate on the held-out test set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds_p)
print(f"Test accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}")

y_true, y_pred, all_images = [], [], []
for images, labels in test_ds_p:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    all_images.extend(images.numpy())

y_true, y_pred = np.array(y_true), np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()


In [ ]:
# Inspect misclassified examples — the fastest way to spot systematic
# confusions (e.g. "frost" vs "glaze" vs "rime" are visually very close —
# don't be surprised if those three account for most of the errors).
wrong_idx = np.where(y_true != y_pred)[0]
print(f"{len(wrong_idx)} / {len(y_true)} misclassified")

show_n = min(10, len(wrong_idx))
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, idx in zip(axes.flat, wrong_idx[:show_n]):
    img = all_images[idx]
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    ax.imshow(img)
    ax.set_title(f"true: {class_names[y_true[idx]]}\npred: {class_names[y_pred[idx]]}", fontsize=9)
    ax.axis('off')
for ax in axes.flat[show_n:]:
    ax.axis('off')
plt.tight_layout(); plt.show()


## 12 — Grad-CAM: what is the model actually looking at?

Worth including in your presentation's "Tech Highlights" section — it's concrete evidence
the model is learning real weather cues (sky texture, cloud cover) rather than something
spurious like watermark artifacts or photo aspect ratio.

In [ ]:
# NOTE: the classic Grad-CAM recipe (build a new Model reusing an
# intermediate-layer output tensor) breaks under Keras 3 when that layer is
# itself a nested Functional submodel (our EfficientNetB0 backbone) — you'll
# hit a KeyError deep in Keras's graph-tracing code if you try it. This version
# sidesteps that entirely: two manual forward passes inside one GradientTape,
# no new merged Model graph required.

def make_gradcam_heatmap(img_array, model, base_model, base_layer_name):
    layer_names = [l.name for l in model.layers]
    base_idx = layer_names.index(base_layer_name)
    head_layers = model.layers[base_idx + 1:]  # everything after the backbone: GAP, Dropout, Dense...

    with tf.GradientTape() as tape:
        feature_maps = base_model(img_array, training=False)
        tape.watch(feature_maps)
        x = feature_maps
        for layer in head_layers:
            x = layer(x, training=False)
        predictions = x
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, feature_maps)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    feature_maps = feature_maps[0]
    heatmap = feature_maps @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)


base_layer_name = base_model.name
sample_batch = next(iter(test_ds_p.take(1)))[0][:4]

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for i in range(4):
    img_array = tf.expand_dims(sample_batch[i], 0)
    heatmap, pred_idx = make_gradcam_heatmap(img_array, model, base_model, base_layer_name)
    orig = sample_batch[i].numpy()
    orig = (orig - orig.min()) / (orig.max() - orig.min() + 1e-8)

    axes[0, i].imshow(orig); axes[0, i].set_title(f"pred: {class_names[pred_idx]}"); axes[0, i].axis('off')
    axes[1, i].imshow(orig); axes[1, i].imshow(heatmap, cmap='jet', alpha=0.45); axes[1, i].axis('off')
plt.tight_layout(); plt.show()


## 13 — Export for deployment

Produces three artifacts:
1. `weathergpt_vision.keras` — full model, for the backend/FastAPI route
2. `weathergpt_vision.tflite` — float16-quantized, for on-device inference in the Flutter app
3. `labels.txt` — class index → name mapping, needed by both consumers

In [ ]:
model.save('weathergpt_vision.keras')

with open('labels.txt', 'w') as f:
    f.write('\n'.join(class_names))

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open('weathergpt_vision.tflite', 'wb') as f:
    f.write(tflite_model)

import os as _os
print(f"Keras model:  {_os.path.getsize('weathergpt_vision.keras') / 1e6:.2f} MB")
print(f"TFLite model: {_os.path.getsize('weathergpt_vision.tflite') / 1e6:.2f} MB")


In [ ]:
# Sanity check: TFLite predictions should closely match the Keras model's.
interpreter = tf.lite.Interpreter(model_path='weathergpt_vision.tflite')
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample_x, sample_y = next(iter(test_ds_p.take(1)))
sample_img = tf.expand_dims(sample_x[0], 0)

keras_pred = model.predict(sample_img, verbose=0)[0]

interpreter.set_tensor(input_details[0]['index'], sample_img.numpy().astype(input_details[0]['dtype']))
interpreter.invoke()
tflite_pred = interpreter.get_tensor(output_details[0]['index'])[0]

print("Keras  top prediction:", class_names[np.argmax(keras_pred)], f"({keras_pred.max():.3f})")
print("TFLite top prediction:", class_names[np.argmax(tflite_pred)], f"({tflite_pred.max():.3f})")
print("Max absolute difference across all classes:", np.abs(keras_pred - tflite_pred).max())


## 14 — Download the trained artifacts

In [ ]:
import zipfile

with zipfile.ZipFile('weathergpt_vision_artifacts.zip', 'w') as zf:
    for fname in ['weathergpt_vision.keras', 'weathergpt_vision.tflite', 'labels.txt', 'training_log.csv']:
        if os.path.exists(fname):
            zf.write(fname)

files.download('weathergpt_vision_artifacts.zip')
# Also consider saving a copy to Google Drive so a Colab disconnect doesn't lose your run:
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copy('weathergpt_vision_artifacts.zip', '/content/drive/MyDrive/')


## 15 — Integrating this into WeatherGPT

**Option A — On-device (recommended for this project):** add `tflite_flutter` to the Flutter
app, bundle `weathergpt_vision.tflite` + `labels.txt` as assets, run inference locally when a
user attaches/takes a photo in chat. Works with zero connectivity — a real advantage for a
disaster-management tool in low-network regions. Preprocessing in Dart must match this
notebook exactly: resize to 224×224, then apply the same `preprocess_input` scaling used above
(EfficientNet: scales to roughly [-1, 1]; MobileNetV2: same range) — a mismatch here is the
single most common cause of a model that scores well in Colab but looks random on-device.

**Option B — Backend-served:** use the included `serve_model.py` (FastAPI) to expose
`POST /api/vision/classify-weather` — add this to the API contract doc from the mobile app
handoff alongside `/api/chat/query`. Heavier on the backend, but simpler to update the model
without shipping a new app build.

## 16 — Next steps for Members 3 & 4
- This Kaggle dataset is global stock photography — accuracy on real Indian sky/weather
  photos (different haze levels, monsoon cloud types, etc.) will likely be somewhat lower.
  If time allows, collect a small validation set of India-specific photos and re-check
  per-class accuracy before trusting this in the live demo.
- `frost` / `glaze` / `rime` are visually near-identical even to a human eye — check the
  confusion matrix in Section 11; it may be worth merging these three into a single
  `icy-conditions` class for a cleaner demo.
- Try `EfficientNetB3` if Colab compute allows — usually another 2-4% test accuracy for
  roughly 3x the inference cost, worth it only for the server-served option, not on-device.
